In [2]:
import numpy as np, pandas as pd, glob
NB = glob.glob('/kaggle/input/notebooks/aadiparkhi/03a*')[0]
BK = ['resnet18', 'resnet50', 'convnext', 'dino']
D = {}
for d in ['places', 'bdd']:
    z = np.load(f'{NB}/feats_{d}.npz'); m = pd.read_csv(f'{NB}/meta_{d}.csv')
    D[d] = {'meta': m, **{k: {'emb': z[f'{k}_emb'].astype(np.float32), 'log': z[f'{k}_log']} for k in BK}}
    print(d, len(m), {k: D[d][k]['emb'].shape for k in BK})

rows = []
for d in D:
    m = D[d]['meta']; c = (m.set == 'corrupt').values
    for k in BK:
        L = D[d][k]['log']
        hits = {'scene': L.argmax(1) == m.label.values} if d == 'places' else {'cond': L[:, :11].argmax(1) == m.cond_id.values, 'person': L[:, 11:].argmax(1) == m.person.values}
        for h, ok in hits.items():
            s = pd.Series(ok[c]).groupby(m.severity.values[c]).mean().round(3)
            rows.append({'domain': d, 'head': h, 'backbone': k, **{f's{i}': s[i] for i in range(6)}})
print(pd.DataFrame(rows).to_string(index=False))

places 36984 {'resnet18': (36984, 512), 'resnet50': (36984, 2048), 'convnext': (36984, 768), 'dino': (36984, 768)}
bdd 36046 {'resnet18': (36046, 512), 'resnet50': (36046, 2048), 'convnext': (36046, 768), 'dino': (36046, 768)}
domain   head backbone    s0    s1    s2    s3    s4    s5
places  scene resnet18 0.596 0.412 0.308 0.244 0.186 0.149
places  scene resnet50 0.711 0.545 0.448 0.356 0.269 0.196
places  scene convnext 0.763 0.652 0.573 0.500 0.405 0.318
places  scene     dino 0.760 0.688 0.610 0.528 0.429 0.333
   bdd   cond resnet18 0.646 0.435 0.372 0.313 0.267 0.243
   bdd person resnet18 0.891 0.824 0.815 0.795 0.787 0.782
   bdd   cond resnet50 0.639 0.517 0.459 0.400 0.337 0.285
   bdd person resnet50 0.923 0.878 0.858 0.840 0.826 0.814
   bdd   cond convnext 0.666 0.540 0.477 0.424 0.350 0.289
   bdd person convnext 0.901 0.872 0.847 0.825 0.813 0.811
   bdd   cond     dino 0.691 0.565 0.507 0.455 0.386 0.329
   bdd person     dino 0.933 0.887 0.869 0.847 0.833 0.821


In [3]:
import time, torch, torch.nn as nn
from sklearn.decomposition import PCA
rng = np.random.default_rng(0); torch.manual_seed(0)
K = 32

def ks_stat(X, S):
    B, n = len(X), len(S); Xs = np.sort(X, 0)
    Fr = np.stack([np.searchsorted(S[:, j], Xs[:, j], side='right') for j in range(K)], 1) / n
    up, lo = np.arange(1, B + 1)[:, None] / B, np.arange(B)[:, None] / B
    return np.maximum(np.abs(up - Fr), np.abs(lo - Fr)).max(0).mean()

def psi(X, E, pr):
    out = 0
    for j in range(K):
        pb = np.clip(np.bincount(np.searchsorted(E[j], X[:, j]), minlength=10) / len(X), 1e-4, None)
        out += ((pb - pr[j]) * np.log(pb / pr[j])).sum()
    return out / K

def train_ae(z, fit):
    x = torch.tensor(z, dtype=torch.float32); tr = x[torch.tensor(fit)]
    ae = nn.Sequential(nn.Linear(x.shape[1], 256), nn.ReLU(), nn.Linear(256, 32), nn.ReLU(),
                       nn.Linear(32, 256), nn.ReLU(), nn.Linear(256, x.shape[1]))
    opt = torch.optim.Adam(ae.parameters(), 1e-3)
    for _ in range(60):
        for b in torch.randperm(len(tr)).split(256):
            opt.zero_grad(); ((ae(tr[b]) - tr[b]) ** 2).mean().backward(); opt.step()
    with torch.no_grad(): return ((ae(x) - x) ** 2).mean(1).numpy()

def prep(d, k):
    m, e = D[d]['meta'], D[d][k]['emb']
    v = (m.set == 'val').values
    z = (e - e[v].mean(0)) / (e[v].std(0) + 1e-6)
    P = PCA(K, random_state=0).fit(z[v]).transform(z)
    R = P[v]; S = np.sort(R, 0)
    E = [np.quantile(R[:, j], np.linspace(0.1, 0.9, 9)) for j in range(K)]
    pr = np.stack([np.clip(np.bincount(np.searchsorted(E[j], R[:, j]), minlength=10) / len(R), 1e-4, None) for j in range(K)])
    return P, S, E, pr, train_ae(z, m.set.isin(['val', 'train']).values)

def score(b, P, S, E, pr, err):
    return np.array([ks_stat(P[b], S), psi(P[b], E, pr), err[b].mean()])

def batches(rows, B, nb):
    return [rng.choice(rows, B) for _ in range(nb)]

def evaluate(d, k, B, nb=10, prepped=None):
    m = D[d]['meta']; pp = prepped or prep(d, k)
    sc = lambda b: score(b, *pp)
    cn = sorted(set(m.corruption[m.set == 'corrupt']) - {'clean'})
    tst = np.where((m.set == 'test').values)[0]; rng.shuffle(tst); h = len(tst) // 2
    Ncal = np.array([sc(b) for b in batches(tst[:h], B, 200)])
    Nevl = np.array([sc(b) for b in batches(tst[h:], B, 200)])
    thr = np.quantile(Ncal, 0.99, axis=0)
    cols = lambda pre, v: {f'{pre}_{n}': x for n, x in zip(['ks', 'psi', 'ae'], v)}
    c0 = np.where(((m.set == 'corrupt') & (m.severity == 0)).values)[0]
    rows = [dict(domain=d, bb=k, B=B, sev=-1, **cols('det', (Nevl > thr).mean(0))),
            dict(domain=d, bb=k, B=B, sev=0, **cols('det', (np.array([sc(b) for b in batches(c0, B, 50)]) > thr).mean(0)))]
    for s in range(1, 6):
        X = np.array([sc(b) for c in cn for b in batches(np.where(((m.set == 'corrupt') & (m.severity == s) & (m.corruption == c)).values)[0], B, nb)])
        auc = [(X[:, j][:, None] > Nevl[:, j][None]).mean() + 0.5 * (X[:, j][:, None] == Nevl[:, j][None]).mean() for j in range(3)]
        rows.append(dict(domain=d, bb=k, B=B, sev=s, **cols('det', (X > thr).mean(0)), **cols('auc', auc)))
    return pd.DataFrame(rows)

t = time.time()
print(evaluate('places', 'dino', 100).round(3).to_string(index=False)); print(round(time.time() - t), 's')

domain   bb   B  sev  det_ks  det_psi  det_ae  auc_ks  auc_psi  auc_ae
places dino 100   -1   0.005    0.080   0.015     NaN      NaN     NaN
places dino 100    0   0.140    0.320   0.000     NaN      NaN     NaN
places dino 100    1   0.820    0.840   0.767   0.968    0.947   0.927
places dino 100    2   0.893    0.947   0.833   0.990    0.977   0.972
places dino 100    3   0.953    0.980   0.907   0.997    0.992   0.983
places dino 100    4   0.993    0.980   0.953   1.000    0.996   0.995
places dino 100    5   1.000    1.000   1.000   1.000    1.000   1.000
19 s


In [4]:
def evaluate2(d, k, B, nb=10, pp=None):
    m = D[d]['meta']; pp = pp or prep(d, k)
    ns = lambda pool, n: np.array([score(b, *pp) for b in batches(pool, B, n)])
    cn = sorted(set(m.corruption[m.set == 'corrupt']) - {'clean'})
    tst = np.where((m.set == 'test').values)[0]; rng.shuffle(tst); h = len(tst) // 2
    c0 = np.where(((m.set == 'corrupt') & (m.severity == 0)).values)[0]
    thr_f = np.quantile(ns(tst[:h], 400), 0.99, axis=0)          # fresh-data threshold
    N1, N2 = ns(c0, 400), ns(c0, 400)                              # paired nulls: clean copies of the same base images
    thr_p = np.quantile(N1, 0.99, axis=0)
    cols = lambda pre, v: {f'{pre}_{n}': x for n, x in zip(['ks', 'psi', 'ae'], v)}
    row = lambda kind, **kw: dict(domain=d, bb=k, B=B, kind=kind, **kw)
    rows = [row('far_fresh', **cols('det', (ns(tst[h:], 400) > thr_f).mean(0))),
            row('far_copies', **cols('det', (ns(c0, 200) > thr_f).mean(0))),
            row('far_paired', **cols('det', (N2 > thr_p).mean(0)))]
    for s in range(1, 6):
        X = np.array([score(b, *pp) for c in cn for b in batches(np.where(((m.set == 'corrupt') & (m.severity == s) & (m.corruption == c)).values)[0], B, nb)])
        auc = [(X[:, j][:, None] > N2[:, j][None]).mean() + 0.5 * (X[:, j][:, None] == N2[:, j][None]).mean() for j in range(3)]
        rows.append(row(f'sev{s}', **cols('det', (X > thr_p).mean(0)), **cols('auc', auc)))
    return pd.DataFrame(rows)

out = []
for d in D:
    for k in BK:
        pp = prep(d, k)
        for B in [50, 100, 200]:
            out.append(evaluate2(d, k, B, pp=pp))
        print(d, k, 'done')
res = pd.concat(out); res.to_csv('/kaggle/working/detect_results.csv', index=False)
for d in D:
    r = res[(res.domain == d) & (res.kind == 'sev1')]
    for p in ['det', 'auc']:
        print(d, p, 'severity 1'); print(r.pivot(index='bb', columns='B', values=[f'{p}_ks', f'{p}_psi', f'{p}_ae']).round(2).to_string())
    print(d, 'false alarms'); print(res[(res.domain == d) & res.kind.str.startswith('far')].pivot_table(index=['kind', 'bb'], columns='B', values=['det_ks', 'det_psi', 'det_ae']).round(2).to_string())

places resnet18 done
places resnet50 done
places convnext done
places dino done
bdd resnet18 done
bdd resnet50 done
bdd convnext done
bdd dino done
places det severity 1
         det_ks             det_psi             det_ae            
B           50    100   200     50    100   200    50    100   200
bb                                                                
convnext   0.70  0.83  0.90    0.50  0.79  0.84   0.79  0.83  0.89
dino       0.53  0.73  0.81    0.36  0.59  0.71   0.61  0.74  0.79
resnet18   0.81  0.85  0.94    0.79  0.84  0.91   0.09  0.14  0.19
resnet50   0.90  0.93  0.93    0.68  0.89  0.92   0.43  0.55  0.61
places auc severity 1
         auc_ks             auc_psi             auc_ae            
B           50    100   200     50    100   200    50    100   200
bb                                                                
convnext   0.92  0.95  0.98    0.87  0.92  0.96   0.94  0.95  0.96
dino       0.88  0.92  0.95    0.79  0.87  0.92   0.90  0.92  0.94
resn

In [5]:
PP = {(d, k): prep(d, k) for d in D for k in BK}
def per_corr(d, k, B=100, sev=1, nb=20):
    m = D[d]['meta']; pp = PP[(d, k)]
    ns = lambda pool, n: np.array([score(b, *pp) for b in batches(pool, B, n)])
    c0 = np.where(((m.set == 'corrupt') & (m.severity == 0)).values)[0]
    thr = np.quantile(ns(c0, 400), 0.99, axis=0)
    cn = sorted(set(m.corruption[m.set == 'corrupt']) - {'clean'})
    return pd.DataFrame({c: (ns(np.where(((m.set == 'corrupt') & (m.severity == sev) & (m.corruption == c)).values)[0], nb) > thr).mean(0) for c in cn}, index=['ks', 'psi', 'ae']).T
for d in D:
    t = pd.concat({k: per_corr(d, k) for k in BK}, axis=1)
    print(d); print(t.round(2).to_string())

places
                  resnet18             resnet50             convnext              dino            
                        ks   psi    ae       ks   psi    ae       ks   psi    ae    ks   psi    ae
brightness             0.0  0.00  0.00     0.05  0.05  0.05     0.00  0.00  0.00  0.00  0.00  0.00
contrast               1.0  1.00  0.00     1.00  1.00  0.00     0.95  0.30  0.60  0.35  0.05  0.10
defocus_blur           1.0  1.00  0.25     1.00  1.00  1.00     1.00  0.90  1.00  1.00  0.75  1.00
elastic_transform      1.0  1.00  0.25     1.00  1.00  1.00     1.00  1.00  1.00  1.00  1.00  1.00
fog                    1.0  1.00  0.00     1.00  1.00  0.00     0.95  0.90  0.95  1.00  0.85  0.95
frost                  1.0  1.00  0.20     1.00  1.00  0.15     1.00  1.00  1.00  1.00  0.95  1.00
gaussian_noise         1.0  1.00  0.55     1.00  0.95  0.60     1.00  1.00  1.00  0.35  0.10  0.95
glass_blur             1.0  1.00  0.00     1.00  1.00  1.00     1.00  1.00  1.00  1.00  1.00  1.00
imp

In [6]:
def drops(d, sev=1):
    m = D[d]['meta']; c = (m.set == 'corrupt').values; out = {}
    for k in BK:
        L = D[d][k]['log']
        ok = (L.argmax(1) == m.label.values) if d == 'places' else (L[:, :11].argmax(1) == m.cond_id.values)
        s = pd.Series(ok[c]).groupby([m.corruption.values[c], m.severity.values[c]]).mean()
        out[k] = pd.Series({n: s[('clean', 0)] - s[(n, sev)] for n in sorted(set(m.corruption[c]) - {'clean'})})
    return pd.DataFrame(out)
for d in D:
    print(d, 'accuracy drop at severity 1'); print(drops(d).round(2).to_string())

places accuracy drop at severity 1
                   resnet18  resnet50  convnext  dino
brightness             0.02      0.01      0.01  0.00
contrast               0.18      0.10      0.04  0.01
defocus_blur           0.27      0.09      0.10  0.07
elastic_transform      0.21      0.44      0.32  0.14
fog                    0.17      0.13      0.07  0.07
frost                  0.15      0.17      0.11  0.08
gaussian_noise         0.27      0.16      0.11  0.05
glass_blur             0.21      0.24      0.16  0.14
impulse_noise          0.34      0.29      0.07  0.09
jpeg_compression       0.05      0.04      0.08  0.03
motion_blur            0.13      0.12      0.05  0.05
pixelate               0.06      0.09      0.13  0.02
shot_noise             0.27      0.19      0.12  0.08
snow                   0.21      0.24      0.12  0.13
zoom_blur              0.22      0.17      0.18  0.13
bdd accuracy drop at severity 1
                   resnet18  resnet50  convnext  dino
brightness     

In [7]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

def ks_vec(X, S):
    B, n = len(X), len(S); Xs = np.sort(X, 0)
    Fr = np.stack([np.searchsorted(S[:, j], Xs[:, j], side='right') for j in range(K)], 1) / n
    up, lo = np.arange(1, B + 1)[:, None] / B, np.arange(B)[:, None] / B
    return np.maximum(np.abs(up - Fr), np.abs(lo - Fr)).max(0)

def correct(d, k):
    m, L = D[d]['meta'], D[d][k]['log']
    return (L.argmax(1) == m.label.values) if d == 'places' else (L[:, :11].argmax(1) == m.cond_id.values)

def build(d, k, B=100, nb=30, seed=0):
    r = np.random.default_rng(seed); m = D[d]['meta']; P, S, E, pr, err = PP[(d, k)]; ok = correct(d, k)
    cp = np.where((m.set == 'corrupt').values)[0]; bid, sev, cor = m.bid.values, m.severity.values, m.corruption.values
    i0 = cp[sev[cp] == 0]; ck = np.zeros(bid[cp].max() + 1); ck[bid[i0]] = ok[i0]
    half = r.permutation(bid[cp].max() + 1) % 2                      # two disjoint halves of the base images
    rows = []
    for c in sorted(set(cor[cp])):
        for s in sorted(set(sev[cp][cor[cp] == c])):
            pool = cp[(cor[cp] == c) & (sev[cp] == s)]
            for h in (0, 1):
                ph = pool[half[bid[pool]] == h]
                for _ in range(nb):
                    b = r.choice(ph, B); X = P[b]; kv = ks_vec(X, S)
                    f = np.concatenate([X.mean(0), X.std(0), kv, [kv.mean(), psi(X, E, pr), err[b].mean()]])
                    rows.append([*f, c, s, h, ck[bid[b]].mean() - ok[b].mean()])
    return pd.DataFrame(rows, columns=[f'f{i}' for i in range(len(rows[0]) - 4)] + ['cause', 'sev', 'split', 'dloss'])

res = []
for d in D:
    for k in BK:
        df = build(d, k); F = [c for c in df.columns if c.startswith('f')]
        tr, te = df[df.split == 0], df[df.split == 1]
        a, b = tr[tr.cause != 'clean'], te[te.cause != 'clean']
        clf = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(a[F], a.cause)
        reg = RandomForestRegressor(300, random_state=0, n_jobs=-1).fit(tr[F], tr.dloss)
        p = reg.predict(te[F])
        res.append(dict(domain=d, bb=k, attr_acc=(clf.predict(b[F]) == b.cause).mean(), r2=r2_score(te.dloss, p), mae=mean_absolute_error(te.dloss, p)))
        print(res[-1])
print(pd.DataFrame(res).round(3).to_string(index=False))

{'domain': 'places', 'bb': 'resnet18', 'attr_acc': np.float64(0.952), 'r2': 0.8633596571203379, 'mae': 0.048005584795321646}
{'domain': 'places', 'bb': 'resnet50', 'attr_acc': np.float64(0.9346666666666666), 'r2': 0.8443238129962368, 'mae': 0.05909779239766082}
{'domain': 'places', 'bb': 'convnext', 'attr_acc': np.float64(0.9493333333333334), 'r2': 0.8847899543524619, 'mae': 0.04788801169590644}
{'domain': 'places', 'bb': 'dino', 'attr_acc': np.float64(0.9475555555555556), 'r2': 0.8796579703639151, 'mae': 0.050130453216374264}
{'domain': 'bdd', 'bb': 'resnet18', 'attr_acc': np.float64(0.9977777777777778), 'r2': 0.5325966246618905, 'mae': 0.0893032894736842}
{'domain': 'bdd', 'bb': 'resnet50', 'attr_acc': np.float64(0.9928888888888889), 'r2': 0.6822706974123545, 'mae': 0.06055377192982458}
{'domain': 'bdd', 'bb': 'convnext', 'attr_acc': np.float64(0.9951111111111111), 'r2': 0.7799938739027629, 'mae': 0.05365998538011693}
{'domain': 'bdd', 'bb': 'dino', 'attr_acc': np.float64(0.995555555

In [8]:
def run(d, k, seed):
    df = build(d, k, seed=seed); F = [c for c in df.columns if c.startswith('f')]
    tr, te = df[df.split == 0], df[df.split == 1]
    a, b = tr[tr.cause != 'clean'], te[te.cause != 'clean']
    clf = RandomForestClassifier(300, random_state=0, n_jobs=-1).fit(a[F], a.cause)
    p = RandomForestRegressor(300, random_state=0, n_jobs=-1).fit(tr[F], tr.dloss).predict(te[F])
    mu = tr.groupby(['cause', 'sev']).dloss.mean()                       # oracle: knows the true condition, not the batch
    o = np.array([mu[(c, s)] for c, s in zip(te.cause, te.sev)])
    return dict(attr=(clf.predict(b[F]) == b.cause).mean(), r2=r2_score(te.dloss, p), mae=mean_absolute_error(te.dloss, p),
                o_r2=r2_score(te.dloss, o), o_mae=mean_absolute_error(te.dloss, o))

rows = [dict(domain=d, bb=k, seed=s, **run(d, k, s)) for d in D for k in BK for s in (1, 2, 3)]
print(pd.DataFrame(rows).drop(columns='seed').groupby(['domain', 'bb']).agg(['mean', 'std']).round(3).to_string())

                  attr            r2           mae          o_r2         o_mae       
                  mean    std   mean    std   mean    std   mean    std   mean    std
domain bb                                                                            
bdd    convnext  0.991  0.004  0.767  0.047  0.058  0.007  0.770  0.042  0.057  0.005
       dino      0.990  0.003  0.803  0.034  0.055  0.007  0.788  0.041  0.057  0.008
       resnet18  0.986  0.007  0.830  0.026  0.054  0.004  0.830  0.028  0.054  0.004
       resnet50  0.981  0.007  0.595  0.247  0.074  0.024  0.610  0.246  0.072  0.024
places convnext  0.937  0.004  0.878  0.031  0.050  0.005  0.878  0.033  0.049  0.005
       dino      0.937  0.013  0.864  0.050  0.054  0.010  0.853  0.067  0.056  0.013
       resnet18  0.939  0.014  0.795  0.025  0.060  0.005  0.789  0.042  0.060  0.007
       resnet50  0.940  0.014  0.876  0.025  0.055  0.006  0.877  0.013  0.054  0.003
